<a href="https://colab.research.google.com/github/superv13/NLP_Assignment-02/blob/main/DependencyParser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
sentence = ["I", "parsed", "this", "sentence", "correctly"]
sentence_text = " ".join(sentence)

# Process the sentence with spaCy
doc = nlp(sentence_text)

# Print dependency parsing results
print("Dependency parsing using spaCy:")
for token in doc:
    print(f"{token.text:<10} {token.dep_:<10} {token.head.text:<10} {spacy.explain(token.dep_)}")

Dependency parsing using spaCy:
I          nsubj      parsed     nominal subject
parsed     ROOT       parsed     root
this       det        sentence   determiner
sentence   dobj       parsed     direct object
correctly  advmod     parsed     adverbial modifier


In [ ]:
def generate_transitions_from_spacy_parse(sentence_tokens):
    transitions = []
    sentence_text = " ".join(sentence_tokens)
    doc = nlp(sentence_text)

    # Map spaCy's token objects to their texts for easier comparison
    spacy_token_map = {token.text: token for token in doc}

    # Helper to check if a dependency (dependent_text, head_text) exists in spaCy's parse
    # This function uses the `spacy_token_map` to get the actual spaCy token object
    # and verifies its head.
    def is_dependent_of(dependent_text, head_text):
        if dependent_text == "ROOT": # Special case for the artificial ROOT dependent
            # Check if the head_text is indeed the spaCy identified ROOT token
            spacy_root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
            return spacy_root_token and spacy_root_token.text == head_text
        elif dependent_text in spacy_token_map:
            token = spacy_token_map[dependent_text]
            return token.head.text == head_text
        return False

    # Simulate PartialParse's stack and buffer for decision making
    # This is a simplified internal simulation, not the actual PartialParse object
    sim_stack = ["ROOT"]
    sim_buffer = list(sentence_tokens)
    current_dependencies = set() # To store (head, dependent) as strings to avoid duplicate arcs

    # Heuristic for generating transitions:
    # This implements a greedy shift-reduce strategy. It tries to resolve arcs
    # between the top two stack elements whenever possible, guided by spaCy's parse.
    # If no arc can be formed, it shifts a token from the buffer to the stack.
    # This is a simplified approach for demonstration and not a full parsing algorithm.
    while len(sim_buffer) > 0 or len(sim_stack) > 2:
        made_transition = False

        # Try LEFT-ARC: stack[-2] is dependent of stack[-1]
        if len(sim_stack) >= 2:
            s1_text = sim_stack[-1]  # Potential Head
            s2_text = sim_stack[-2]  # Potential Dependent

            if is_dependent_of(s2_text, s1_text) and (s1_text, s2_text) not in current_dependencies:
                transitions.append(("LEFT-ARC", f"{s1_text} is head of {s2_text}"))
                current_dependencies.add((s1_text, s2_text))
                sim_stack.pop(-2) # Remove the dependent (s2_text)
                made_transition = True
                continue # Re-evaluate state after a transition

        # Try RIGHT-ARC: stack[-1] is dependent of stack[-2]
        if len(sim_stack) >= 2:
            s1_text = sim_stack[-1]  # Potential Dependent
            s2_text = sim_stack[-2]  # Potential Head

            if is_dependent_of(s1_text, s2_text) and (s2_text, s1_text) not in current_dependencies:
                transitions.append(("RIGHT-ARC", f"{s2_text} is head of {s1_text}"))
                current_dependencies.add((s2_text, s1_text))
                sim_stack.pop() # Remove the dependent (s1_text)
                made_transition = True
                continue # Re-evaluate state after a transition

        # If no arc was made, try to SHIFT
        if len(sim_buffer) > 0:
            next_token = sim_buffer.pop(0)
            transitions.append(("SHIFT", next_token))
            sim_stack.append(next_token)
            made_transition = True
            continue # Re-evaluate state after a transition

        # If no transition was made in an iteration and we are stuck, break to avoid infinite loop
        if not made_transition:
            break

    # Final ROOT attachment: If stack contains ['ROOT', main_root_word], apply LEFT-ARC.
    # The PartialParse's LEFT-ARC (head=stack[-1], dependent=stack[-2]) means (main_root_word, ROOT).
    if len(sim_stack) == 2 and sim_stack[0] == "ROOT":
        root_of_sentence_text = sim_stack[1]
        if (root_of_sentence_text, "ROOT") not in current_dependencies: # Ensure we don't add duplicate
            # This is the actual dependency formed by PartialParse.left_arc() when stack is ['ROOT', root_of_sentence_text]
            transitions.append(("LEFT-ARC", f"{root_of_sentence_text} is head of ROOT"))

    return transitions

In [ ]:
def build_and_print_dependency_tree(dependencies):
    # Create a dictionary to store parent-child relationships
    # Each key is a head, and its value is a list of its dependents
    tree = {}
    for head, dependent in dependencies:
        if head not in tree:
            tree[head] = []
        tree[head].append(dependent)

    # Identify the root of the tree (it's 'parsed' in this specific output, or 'ROOT' is usually the conceptual root)
    # For this specific output, 'ROOT' is a dependent of 'parsed', implying 'parsed' is the sentence root.
    # Let's find the true root (the one that is not dependent on any other word, except ROOT)
    all_dependents = {dep for _, dep in dependencies}
    all_heads = {head for head, _ in dependencies}
    # The actual ROOT of the parse is usually the word that the 'ROOT' pseudo-token points to.
    # In our case, it's 'parsed' as seen from ('parsed', 'ROOT').
    # So, we should start printing from 'parsed'.
    sentence_root = None
    for head, dependent in dependencies:
        if dependent == 'ROOT':
            sentence_root = head
            break

    if not sentence_root:
        print("Could not determine the sentence root from dependencies.")
        return

    def print_node(node, indent, is_last_child, prefix=""):
        print(f"{indent}{prefix}{node}")
        children = tree.get(node, [])

        # Sort children for consistent output, if desired
        children.sort()

        for i, child in enumerate(children):
            new_prefix = "└─ " if i == len(children) - 1 else "├─ "
            new_indent = indent + ("    " if is_last_child else "│   ")
            print_node(child, new_indent, i == len(children) - 1, new_prefix)

    print("\n--- Dependency Tree ---")
    print_node(sentence_root, "", True)

In [ ]:
class PartialParse:

    def __init__(self, sentence):

        self.stack = ["ROOT"]
        self.buffer = list(sentence)
        self.dependencies = []

    def shift(self):
        self.stack.append(self.buffer.pop(0))

    def left_arc(self):
        head = self.stack[-1]
        dependent = self.stack[-2]
        self.dependencies.append((head, dependent))
        self.stack.pop(-2)

    def right_arc(self):
        head = self.stack[-2]
        dependent = self.stack[-1]
        self.dependencies.append((head, dependent))
        self.stack.pop()

    def parse_step(self, transition):

        if transition == "SHIFT":
            self.shift()

        elif transition == "LEFT-ARC":
            self.left_arc()

        elif transition == "RIGHT-ARC":
            self.right_arc()

    def print_state(self):
        print(f"  Stack: {self.stack}")
        print(f"  Buffer: {self.buffer}")
        print(f"  Dependencies: {self.dependencies}")

sentence = ["I", "parsed", "this", "sentence", "correctly"]

In [ ]:
def run_partial_parse_demo(sentence_to_parse, transitions_list):
    print("Initializing PartialParse:")
    pp = PartialParse(sentence_to_parse)
    pp.print_state()

    # Use the provided transitions_list instead of hardcoding
    for transition_type, description in transitions_list:
        print(f"\n--- Performing {transition_type} ({description}) ---")
        pp.parse_step(transition_type)
        pp.print_state()

    return pp

In [ ]:
print("\n--- Demonstrating with spaCy-generated transitions ---")

generated_transitions = generate_transitions_from_spacy_parse(sentence)

# Run the demo with our sentence and the spaCy-generated transitions
final_parser_state_spacy = run_partial_parse_demo(sentence, generated_transitions)

# Print the raw dependencies from the final state
print(f"\nFinal Raw Dependencies (spaCy generated): {final_parser_state_spacy.dependencies}")

# Now, let's use the build_and_print_dependency_tree function with the final dependencies
build_and_print_dependency_tree(final_parser_state_spacy.dependencies)


--- Demonstrating with spaCy-generated transitions ---
Initializing PartialParse:
  Stack: ['ROOT']
  Buffer: ['I', 'parsed', 'this', 'sentence', 'correctly']
  Dependencies: []

--- Performing SHIFT (I) ---
  Stack: ['ROOT', 'I']
  Buffer: ['parsed', 'this', 'sentence', 'correctly']
  Dependencies: []

--- Performing SHIFT (parsed) ---
  Stack: ['ROOT', 'I', 'parsed']
  Buffer: ['this', 'sentence', 'correctly']
  Dependencies: []

--- Performing LEFT-ARC (parsed is head of I) ---
  Stack: ['ROOT', 'parsed']
  Buffer: ['this', 'sentence', 'correctly']
  Dependencies: [('parsed', 'I')]

--- Performing LEFT-ARC (parsed is head of ROOT) ---
  Stack: ['parsed']
  Buffer: ['this', 'sentence', 'correctly']
  Dependencies: [('parsed', 'I'), ('parsed', 'ROOT')]

--- Performing SHIFT (this) ---
  Stack: ['parsed', 'this']
  Buffer: ['sentence', 'correctly']
  Dependencies: [('parsed', 'I'), ('parsed', 'ROOT')]

--- Performing SHIFT (sentence) ---
  Stack: ['parsed', 'this', 'sentence']
  Buffe

In [ ]:
new_sentence = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog", "."]

print(f"\n--- Demonstrating with new sentence: {' '.join(new_sentence)} ---")

# 1. Generate transitions for the new sentence using spaCy
generated_transitions_new = generate_transitions_from_spacy_parse(new_sentence)

# 2. Run the partial parse demo with the new sentence and its generated transitions
final_parser_state_new = run_partial_parse_demo(new_sentence, generated_transitions_new)

# 3. Print the raw dependencies from the final state
print(f"\nFinal Raw Dependencies (spaCy generated for new sentence): {final_parser_state_new.dependencies}")

# 4. Build and print the dependency tree for the new sentence
build_and_print_dependency_tree(final_parser_state_new.dependencies)


--- Demonstrating with new sentence: The quick brown fox jumps over the lazy dog . ---
Initializing PartialParse:
  Stack: ['ROOT']
  Buffer: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
  Dependencies: []

--- Performing SHIFT (The) ---
  Stack: ['ROOT', 'The']
  Buffer: ['quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
  Dependencies: []

--- Performing SHIFT (quick) ---
  Stack: ['ROOT', 'The', 'quick']
  Buffer: ['brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
  Dependencies: []

--- Performing SHIFT (brown) ---
  Stack: ['ROOT', 'The', 'quick', 'brown']
  Buffer: ['fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
  Dependencies: []

--- Performing SHIFT (fox) ---
  Stack: ['ROOT', 'The', 'quick', 'brown', 'fox']
  Buffer: ['jumps', 'over', 'the', 'lazy', 'dog', '.']
  Dependencies: []

--- Performing LEFT-ARC (fox is head of brown) ---
  Stack: ['ROOT', 'The', 'quick', 'fox']
  Buffer: ['jumps', 'over', 'the', 'lazy